# RPM Third-Party Vendor Fraud Analysis

Adds peer-group z-scores and isolation forest on top of SQL-based scoring.  
Prereq: `kn_rpm_202608_fraud.sql` has created all tables.

**Reads**: `tmp_1m.kn_rpm_202608_fraud_scores`  
**Writes**: CSV export + updated signal columns

In [63]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
from snowflake.sqlalchemy import URL
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings("ignore")

engine = create_engine(URL(
    account = "UHG-UHGDWAAS",
    user = "KHANG.NGUYEN@UHC.COM",
    authenticator = "externalbrowser",
    role = "AZU_SDRP_VING_PRD_DEVELOPER_ROLE",
    warehouse = "VING_PRD_MNR_HCE_DATAINFRA_WH",
    database = "VING_PRD_TREND_DB",
    schema = "TMP_1M"
))

In [64]:
## 1. Load SQL-scored data
df = pd.read_sql("select * from tmp_1m.kn_rpm_202608_fraud_scores", engine)
df.columns = df.columns.str.lower()

# dominant market per TIN from the detail table
market = pd.read_sql("""
    select prov_tin, market_fnl as market
    from (
        select prov_tin, market_fnl, count(*) as n,
               row_number() over (partition by prov_tin order by count(*) desc) as rn
        from tmp_1m.kn_rpm_202608_detail
        group by prov_tin, market_fnl
    )
    where rn = 1
""", engine)
market.columns = market.columns.str.lower()

df = df.merge(market, on = "prov_tin", how = "left")
print(f"{len(df)} TINs, {df.shape[1]} columns (market fill: {df.market.notna().mean():.1%})")
df.head(3)

 pip install snowflake-connector-python[secure-local-storage]


Initiating login request with your identity provider. Press CTRL+C to abort and try again...
Going to open: https://login.microsoftonline.com/db05faca-c82a-4b9d-b9c5-0f64b6755421/saml2?SAMLRequest=lVJdb9owFP0rkfec2AkfLRZQURAaU0cZHwXtZXJsBywcO7UdQvfr5wSQuodW6oMlyz7nnnPvuf2Hcy6DEzdWaDUAcYRAwBXVTKj9AGzW0%2FAeBNYRxYjUig%2FAG7fgYdi3JJcFHpXuoJb8teTWBb6Qsrj%2BGIDSKKyJFRYrknOLHcWr0c8nnEQIE2u5cV4OXCnMCq91cK7AEFZVFVWtSJs9TBBCEPWgR9WQb%2BCdRPG5RmG001TLG%2BXse%2FpAIoaoXUt4hFdYXImPQl1G8JlKegFZ%2FH29XoSL59UaBKNbd2OtbJlzs%2BLmJCjfLJ8uBqx3UB72oT%2BsIsT%2BOZDIKl1lkhw51XlROl8z8jeYcQal3gs%2FqdlkAIqjYN2t3u6WZJdNGdtO07%2Fn%2BXmWTQl9Phk6f12Mj4%2B7Hy3S%2BrXrbSgIXm65JnWuM2tLPlN1ms4%2FoaQbol6I2uv4DrdinCRR0u38BsHEpykUcQ3zZrnxEeWCGm115rSSQvHGJUtRJyOUhPQ%2BIWE77bEw7dFOiLJuO%2B3edTrtJIZ1Zgm47A1ujJjh16bRh%2B%2B51wWc%2B0xmk4WWgr4FU21y4j6OLI7i5kWwMGugmOdEyBFjhlvro5NSV2PDifN77kzJARxeVP%2Ff9OE%2F&RelayState=ver%3A3-hint%3A260092392366162-ETMsDgAAAaBteZ8ZABRBRVMvQ0JDL1BLQ1M1UGFkZGluZwEAABAAEKYESHtAVD4z0qu%2FdWmQziYAAACQ

,prov_tin,sample_full_nm,clin_nm,hospital_group,sample_delegated_entity,vendor_name_flag,primary_specialty,tadm_prov_spec_cat,provider_state,provider_zip,...,total_signals,dom_relationship,dom_growth,dom_billing_coding,dom_retention,dom_network,total_domains,priority_tier,priority_tier_domain,market
0,383699261,RESPIRATORY SUPPORT SERVICES,None,RESPIRATORY SUPPORT SERVICES,NO DELEGATION,0,34.0,None,OH,44223,...,1,0,0,0,1,0,1,Low,Low,OH
1,463333885,"CARRINGTON, M.D., EDWARD A.",EDWARD A CARRINGTON MD,None,NO DELEGATION,0,999.0,PC_FAMPRAC,AZ,85282,...,0,0,0,0,0,0,0,Clean,Clean,AZ
2,630897654,"MCFADEN, M.D., THOMAS G.",None,MED HELP,NO DELEGATION,0,73.0,PC_FAMPRAC,AL,35209,...,3,0,0,1,0,1,2,High,Medium,AL


In [65]:
## SQL-based signal distribution before adding Python lenses
## Each signal is a binary flag (0/1) computed from p95 thresholds on TINs with 25+ members.
##
## sig_no_prior      — members had no prior E&M/telehealth claims at this TIN (acquiring strangers)
## sig_mbr_multi_tin — members billed by 3+ TINs for RPM (shared across vendor network)
## sig_ramp          — abnormally large month-over-month enrollment spike (sudden billing operation)
## sig_billing_conc  — claims clustered on one calendar day OR heavy weekend billing (batch/automated)
## sig_cpt_bundle    — high 99458 add-on rate OR narrow procedure mix (upcoding/templated billing)
## sig_bill_order    — setup + management billed same day OR management before setup (fabricated)
## sig_no_treat      — members never received treatment management from ANY TIN (device-and-bill)
## sig_churn         — most members leave after 1 month (enroll, bill setup, move on)
## sig_tin_network   — high member/NPI overlap with other TINs (shared vendor shell network)
sql_signals = ["sig_no_prior", "sig_mbr_multi_tin", "sig_ramp", "sig_billing_conc",
               "sig_cpt_bundle", "sig_bill_order", "sig_no_treat",
               "sig_churn", "sig_tin_network"]

print("SQL signal counts:")
print(df[sql_signals].sum().to_string())
print(f"\nSQL priority tier distribution:")
print(df.priority_tier.value_counts().reindex(["High", "Medium", "Low", "Clean"]).to_string())

SQL signal counts:
sig_no_prior         1031
sig_mbr_multi_tin     411
sig_ramp              106
sig_billing_conc     1916
sig_cpt_bundle       1864
sig_bill_order       1485
sig_no_treat          624
sig_churn            1294
sig_tin_network       842

SQL priority tier distribution:
priority_tier
High      1249
Medium    1368
Low       2443
Clean     2850


## 2. Global z-scores

Z-scores are computed against all RPM-billing TINs (no peer grouping). A TIN is flagged if z > 3 on 2+ features. Peer grouping by specialty, size, and practice-type was evaluated (see section 2a) and rejected: no grouping variable reduced within-group variance meaningfully.

In [66]:
z_features = [
    "peak_mom_growth_pct", "growth_ratio",
    "top_dom_pct", "avg_claims_per_mbr_month", "weekend_pct",
    "pct_99458_addon", "setup_ratio", "pct_full_bundle", "proc_hhi",
    "single_month_churn_rate", "median_member_tenure_months",
    "member_state_entropy", "pct_member_state_mismatch",
    "pct_sameday_setup_mgmt", "pct_mgmt_before_setup",
    "pct_no_prior_rel", "mbrs_per_srvc_npi",
    "pct_shared_npis", "max_jaccard"
]

print(f"{len(z_features)} features for z-score computation (global, no peer grouping)")

19 features for z-score computation (global, no peer grouping)


In [67]:
## For each metric, compute how many standard deviations this TIN is from the global mean.
## z > 3 means the TIN is in the extreme tail -- far outside normal for all RPM-billing TINs.
## A TIN flagged on 2+ features is a multivariate outlier, not just extreme on one quirky metric.
##
## Peer grouping (by specialty, size, practice-type) was evaluated and rejected.
## Specialty-based: 6.5% variance reduction. Size-based: -18.4% (worse than global).
## No grouping variable explains meaningful variance in RPM billing metrics.
## See section 2a for the evaluation cells.
def compute_z(series):
    return (series - series.mean()) / max(series.std(), 1e-10)

z_cols = {}
for feat in z_features:
    col_name = f"z_{feat}"
    z_cols[col_name] = compute_z(df[feat])

z_df = pd.DataFrame(z_cols, index = df.index)
df = df.assign(
    n_z_flags = z_df.gt(3).sum(axis = 1),
    sig_peer_zscore = lambda x: (x.n_z_flags >= 2).astype(int)
)

print(f"TINs with any z > 3: {z_df.gt(3).any(axis = 1).sum()}")
print(f"TINs flagged (z > 3 on 2+ features): {df.sig_peer_zscore.sum()}")
print(f"\nz-flag count distribution:")
print(df.n_z_flags.value_counts().sort_index().to_string())

TINs with any z > 3: 714
TINs flagged (z > 3 on 2+ features): 102

z-flag count distribution:
n_z_flags
0    7196
1     612
2      82
3      15
4       3
5       2


In [68]:
# Which features produce the most z > 3 outliers?
print("Features ranked by z > 3 count:")
print(z_df.gt(3).sum().sort_values(ascending = False).to_string())

Features ranked by z > 3 count:
z_pct_member_state_mismatch      284
z_median_member_tenure_months    160
z_weekend_pct                    125
z_member_state_entropy           115
z_mbrs_per_srvc_npi               52
z_growth_ratio                    32
z_setup_ratio                     21
z_peak_mom_growth_pct             19
z_max_jaccard                     12
z_avg_claims_per_mbr_month        12
z_pct_99458_addon                 11
z_top_dom_pct                      0
z_single_month_churn_rate          0
z_pct_full_bundle                  0
z_proc_hhi                         0
z_pct_mgmt_before_setup            0
z_pct_sameday_setup_mgmt           0
z_pct_no_prior_rel                 0
z_pct_shared_npis                  0


## 3. Isolation Forest

Runs on vendor-diagnostic features only (prior relationship, ramp, billing regularity, churn, geography, proc sequencing). Not volume features, which would just flag large health systems. Contamination = 5%.

In [69]:
## Isolation forest: unsupervised anomaly detection on vendor-diagnostic features.
## It builds random trees and measures how quickly each TIN gets isolated.
## TINs that are easy to isolate (short path length) have unusual feature combinations.
## contamination = 0.05 means we flag the most anomalous 5%.
## Volume features (total_mbrs) are excluded — we don't want to just flag large health systems.
iso_features = [
    "pct_no_prior_rel", "peak_mom_growth_pct", "growth_ratio",
    "top_dom_pct", "dom_entropy", "avg_claims_per_mbr_month",
    "pct_99458_addon", "pct_full_bundle", "single_month_churn_rate",
    "median_member_tenure_months", "member_state_entropy",
    "pct_sameday_setup_mgmt", "proc_hhi", "mbrs_per_srvc_npi"
]

X = df[iso_features].copy().fillna(df[iso_features].median())
X_scaled = StandardScaler().fit_transform(X)

iso = IsolationForest(n_estimators = 500, contamination = 0.05, random_state = 42, n_jobs = -1)
iso.fit(X_scaled)

df = df.assign(
    iso_score = iso.decision_function(X_scaled),
    sig_isolation = (iso.predict(X_scaled) == -1).astype(int)
)

print(f"Isolation forest flags (top 5%): {df.sig_isolation.sum()}")
print(f"\nIso score distribution:")
print(df.iso_score.describe().to_string())

Isolation forest flags (top 5%): 396

Iso score distribution:
count    7910.000000
mean        0.078483
std         0.041758
min        -0.193650
25%         0.056355
50%         0.084502
75%         0.109343
max         0.148963


In [70]:
# Overlap between isolation forest and peer z-score flags
print("Overlap matrix (iso flagged x z-score flagged):")
print(pd.crosstab(df.sig_isolation, df.sig_peer_zscore, margins = True))

Overlap matrix (iso flagged x z-score flagged):
sig_peer_zscore     0    1   All
sig_isolation                   
0                7474   40  7514
1                 334   62   396
All              7808  102  7910


## 4. Combined scoring (domain-based)

Groups the 10 signal categories into 6 independent domains so correlated signals don't inflate tier counts. Domains: relationship, growth, billing/coding, retention/geo, network, statistical. Tier cutoffs: High = 4+ domains, Medium = 2-3, Low = 1, Clean = 0.

In [71]:
all_signals = sql_signals + ["sig_peer_zscore", "sig_isolation"]

## Domain-based scoring groups correlated signals so they don't inflate the count.
## E.g., billing regularity and coding concentration often co-occur — they represent
## one concern ("billing practices look suspicious"), not two independent red flags.
##
## 6 independent domains:
## 1. RELATIONSHIP  — are they enrolling strangers or shared across TINs?
##                     (sig_no_prior | sig_mbr_multi_tin)
## 2. GROWTH        — did volume spike suspiciously fast? (sig_ramp)
## 3. BILLING/CODING — are claims templated, upcoded, or sequenced wrong?
##                     (sig_billing_conc | sig_cpt_bundle | sig_bill_order | sig_no_treat)
## 4. RETENTION     — do members churn fast?
##                     (sig_churn)
## 5. NETWORK       — is this TIN connected to other suspicious TINs?
##                     (sig_tin_network)
## 6. STATISTICAL   -- does this TIN look anomalous statistically?
##                     (sig_peer_zscore | sig_isolation)
##                     sig_peer_zscore now uses global z-scores (no peer grouping)
##
## Tier cutoffs: High = 4+ domains, Medium = 2-3, Low = 1, Clean = 0
## A "High" provider has problems in at least 4 of 6 independent areas.
df = df.assign(
    total_signals_all = lambda x: x[all_signals].sum(axis = 1),
    dom_relationship = lambda x: ((x.sig_no_prior == 1) | (x.sig_mbr_multi_tin == 1)).astype(int),
    dom_growth = lambda x: x.sig_ramp,
    dom_billing_coding = lambda x: ((x.sig_billing_conc == 1) | (x.sig_cpt_bundle == 1) | (x.sig_bill_order == 1) | (x.sig_no_treat == 1)).astype(int),
    dom_retention = lambda x: (x.sig_churn == 1).astype(int),
    dom_network = lambda x: x.sig_tin_network,
    dom_statistical = lambda x: ((x.sig_peer_zscore == 1) | (x.sig_isolation == 1)).astype(int),
    total_domains = lambda x: (x.dom_relationship + x.dom_growth + x.dom_billing_coding
                               + x.dom_retention + x.dom_network + x.dom_statistical),
    priority_tier_all = lambda x: np.where(
        x.total_domains >= 4, "High",
        np.where(x.total_domains >= 2, "Medium",
            np.where(x.total_domains >= 1, "Low", "Clean")))
)

print("Final priority tier distribution (6 domains):")
print(df.priority_tier_all.value_counts().reindex(["High", "Medium", "Low", "Clean"]).to_string())

print(f"\nCompare to SQL-only domain tier:")
print(df.priority_tier_domain.value_counts().reindex(["High", "Medium", "Low", "Clean"]).to_string())

print(f"\nRaw signal count distribution (10 lenses):")
print(df.total_signals_all.value_counts().sort_index().to_string())

Final priority tier distribution (6 domains):
priority_tier_all
High       182
Medium    1978
Low       2951
Clean     2799

Compare to SQL-only domain tier:
priority_tier_domain
High       112
Medium    1903
Low       3045
Clean     2850

Raw signal count distribution (10 lenses):
total_signals_all
0    2799
1    2374
2    1373
3     761
4     392
5     172
6      33
7       6


In [72]:
# How many TINs moved tiers after adding Python lenses?
tier_shift = pd.crosstab(df.priority_tier_domain, df.priority_tier_all, margins = True)
tier_shift.index.name = "SQL domain tier"
tier_shift.columns.name = "Final domain tier"
print("Tier migration (SQL 5-domain -> Final 6-domain):")
tier_shift

Tier migration (SQL 5-domain -> Final 6-domain):


Final domain tier,Clean,High,Low,Medium,All
SQL domain tier,,,,,
Clean,2799,0,51,0,2850
High,0,112,0,0,112
Low,0,0,2900,145,3045
Medium,0,70,0,1833,1903
All,2799,182,2951,1978,7910


## 5. Vendor calibration check

Known/suspected vendors should score high. If they don't, the model is missing something.

In [73]:
vendor_cols = ["prov_tin", "market", "hospital_group", "clin_nm", "sample_full_nm",
               "total_mbrs", "total_domains", "priority_tier_all",
               "total_signals", "priority_tier", "sig_peer_zscore", "sig_isolation"]

(
    df
    .query("vendor_name_flag == 1")
    .sort_values("total_mbrs", ascending = False)
    [vendor_cols]
    .head(10)
)

,prov_tin,market,hospital_group,clin_nm,sample_full_nm,total_mbrs,total_domains,priority_tier_all,total_signals,priority_tier,sig_peer_zscore,sig_isolation
342,822612399,TX,TAPESTRY TELEHEALTH,None,BOBBY A MALIK MD,33384,2,Medium,1,Low,1,1
7138,871568225,CT,None,None,"ARANDI, M.D., NAVID R.",3766,4,High,3,High,1,1
3454,852232119,NY,CHESS MEDICINE,None,CHESS MEDICINE,3245,3,Medium,2,Medium,1,1
3930,931345639,IL,None,FREEDMAN RPM MEDICAL,"DAUBER, MARTIN",1449,2,Medium,1,Low,1,1
5735,850603082,AZ,"CHAN, RODRIGO",None,"CHAN, M.D., RODRIGO C.",976,3,Medium,2,Medium,0,1
22,872434668,NJ,None,None,"CROYMANS, DANIEL M.",658,3,Medium,2,Medium,0,1
4769,410855367,MN,OLMSTED MEDICAL CENTER,None,"ABCEJO, M.D., ANDREA V.",269,1,Low,1,Low,0,0
3783,852958439,OH,HEALTH MONITORING SOLUTIONS,JAYANTILAL D BHIMANI MD,"BHIMANI, JAYANTILAL D.",264,3,Medium,2,Medium,0,1
4219,731310891,OK,WARREN CLINIC,None,"BHADRIRAJU, SATISH",245,2,Medium,2,Medium,0,1
3684,561667838,NC,Atrium Health System,None,"DOWNEY, M.D., WILLIAM EDWARD",231,1,Low,1,Low,0,0


## 6. Top investigation targets

High-priority TINs with 10+ members, ranked by total signals then member count.

In [76]:
target_cols = ["prov_tin", "market", "hospital_group", "clin_nm", "sample_full_nm",
               "primary_specialty", "total_mbrs",
               "total_domains", "priority_tier_all"] + all_signals

(
    df
    .query("priority_tier_all == 'High' and total_mbrs >= 25")
    .sort_values(["total_domains", "total_mbrs"], ascending = [False, False])
    [target_cols]
    .head(20)
)

,prov_tin,market,hospital_group,clin_nm,sample_full_nm,primary_specialty,total_mbrs,total_domains,priority_tier_all,sig_no_prior,sig_mbr_multi_tin,sig_ramp,sig_billing_conc,sig_cpt_bundle,sig_bill_order,sig_no_treat,sig_churn,sig_tin_network,sig_peer_zscore,sig_isolation
4755,993099556,GA,None,ANNIE H PARK DO,EXCELSIOR HEALTHCARE PARTNERS,4.0,317,5,High,0,1,1,0,0,1,0,0,1,0,1
2784,582317663,GA,ATHENS HEART CENTER,None,"AGRAWAL, M.D., SUBODH K.",4.0,177,5,High,0,1,1,0,0,0,0,1,1,0,1
5380,923554980,NY,None,None,"FAUCI, ALEXANDER",1.0,100,5,High,1,0,1,0,0,1,0,0,1,0,1
3871,620479367,TN,Methodist Healthcare Memphis,None,METHODIST HOSPITAL GERMANTOWN,52.0,55,5,High,1,0,0,0,0,1,0,1,1,0,1
7447,421477471,IA,Unity Point Health,None,UNITYPOINT AT HOME,72.0,53,5,High,1,0,0,0,1,0,0,1,1,0,1
7138,871568225,CT,None,None,"ARANDI, M.D., NAVID R.",1.0,3766,4,High,1,0,1,0,0,0,1,0,0,1,1
508,841894499,IL,INTEGRATED CLINIX,None,ERIC MILLER MD,4.0,1264,4,High,1,0,0,0,1,0,0,1,0,1,1
6999,861308756,GA,None,OUR COMMUNITY HEALTHCARE SYSTEM,AMINAH D PHELPS MD,1.0,1254,4,High,0,1,0,0,1,0,0,0,1,0,1
1953,611442064,GA,ALIEF GERIATRICS,None,ALIEF GERIATRICS,1.0,931,4,High,0,1,0,0,1,0,0,0,1,1,1
7172,262824776,GA,Piedmont Health System,None,"AARON, M.D., HAYDEN KOEHLER",4.0,492,4,High,0,0,1,0,0,1,0,1,0,1,1


In [77]:
# Signal co-occurrence among High-tier TINs
high = df.query("priority_tier_all == 'High'")
print(f"High-tier TINs: {len(high)}")
print(f"\nSignal prevalence among High-tier:")
print(high[all_signals].mean().sort_values(ascending = False).map("{:.1%}".format).to_string())

High-tier TINs: 182

Signal prevalence among High-tier:
sig_churn            89.0%
sig_no_prior         78.6%
sig_tin_network      75.8%
sig_billing_conc     69.8%
sig_cpt_bundle       66.5%
sig_isolation        47.3%
sig_mbr_multi_tin    22.5%
sig_bill_order       17.0%
sig_no_treat         11.0%
sig_peer_zscore       9.3%
sig_ramp              4.4%


## 7. Evidence columns for interpretability

Each signal flag is binary (fired / didn't fire). To explain *why* a provider was flagged,
we need the actual metric values and peer benchmarks side by side.

**Signal glossary (for writing your narrative):**

| Signal | What it measures | Red flag meaning |
|--------|-----------------|------------------|
| `sig_no_prior` | % of RPM members with zero prior E&M/telehealth claims at this TIN (since 2021) | Provider is enrolling strangers into RPM, not managing existing patients |
| `sig_mbr_multi_tin` | % of TIN's members also receiving RPM from 2+ other TINs (n_tins >= 3) | Members shared across vendor network; same patients billed by multiple practices |
| `sig_ramp` | Largest month-over-month growth spike in RPM member count | Sudden volume explosion suggests a newly stood-up billing operation |
| `sig_billing_conc` | Highest single day-of-month share of all claim lines, OR weekend billing % | Claims clustered on one day or weekends suggests batch/automated billing, not real encounters |
| `sig_cpt_bundle` | 99458 add-on rate relative to 99457 base, OR procedure code concentration (HHI) | Upcoding the 20-min add-on or billing the same handful of codes for everyone |
| `sig_bill_order` | % of members with setup + management on the same day, OR management billed before setup | Clinically impossible: can't have 20 min of monitoring data on the day the device is set up |
| `sig_no_treat` | % of TIN's RPM members who never received treatment management (99457/99458) from ANY TIN | Device-and-bill operation: enroll, supply device, collect payment, skip the clinical follow-through |
| `sig_churn` | % of members active for only 1 month | Enroll, bill setup, move on. Not providing ongoing monitoring |
| `sig_tin_network` | Jaccard member overlap with other TINs, OR % of shared servicing NPIs | Same vendor operating multiple TIN shells to multiply billing |
| `sig_peer_zscore` | z > 3 on 2+ metrics vs all RPM-billing TINs (global z-scores) | Statistical outlier across multiple dimensions simultaneously |
| `sig_isolation` | Isolation forest anomaly score (top 5%) | Multivariate anomaly: combination of features that doesn't look like normal providers |

In [78]:
## Peer benchmarks: median and p95 for each key metric (TINs with 25+ members)
## These are the reference points for "normal" — p95 is the signal threshold

benchmark_metrics = {
    # sig_no_prior: what % of this TIN's RPM members had NO prior claims here?
    "pct_no_prior_rel": "% members with no prior relationship at this TIN",

    # sig_mbr_multi_tin: what % of this TIN's members are billed by 3+ TINs?
    "pct_multi_practice_overlap": "% members also receiving RPM from 2+ other TINs",

    # sig_ramp: how fast did this TIN's RPM volume spike?
    "peak_mom_growth_pct": "largest month-over-month member growth (%)",
    "growth_ratio": "ratio of peak month members to first month members",

    # sig_billing_conc: are claims suspiciously clustered on one day or on weekends?
    "top_dom_pct": "highest single day-of-month share of all claim lines",
    "weekend_pct": "% of claim lines on Saturday/Sunday",

    # sig_cpt_bundle: is this TIN upcoding or using a narrow set of procedure codes?
    "pct_99458_addon": "99458 add-on lines / 99457 base lines (>1 = upcoding)",
    "proc_hhi": "procedure code concentration (HHI, 0-1; 1 = one code only)",
    "pct_full_bundle": "% of members billed all 4 RPM codes (setup+education+mgmt+addon)",

    # sig_churn: do members leave after one month?
    "single_month_churn_rate": "% of members active for exactly 1 month",
    "median_member_tenure_months": "median months a member stays enrolled",

    # sig_geo: are members spread across many states?
    "member_state_entropy": "Shannon entropy of member state distribution (0 = one state)",
    "pct_member_state_mismatch": "% of members in a different state than the provider",

    # sig_bill_order: is setup + management billed on the same day?
    "pct_sameday_setup_mgmt": "% of members with setup and management on the same date",
    "pct_mgmt_before_setup": "% of members with management billed BEFORE device setup",

    # sig_no_treat: do members ever get treatment management?
    "pct_no_treatment_mgmt": "% of members who never received treatment management from any TIN",

    # sig_tin_network: does this TIN share members or NPIs with other TINs?
    "pct_shared_npis": "% of servicing NPIs that also bill under other TINs",
    "max_jaccard": "highest Jaccard similarity of member overlap with another TIN",

    # sig_peer_zscore / sig_isolation: statistical outlier measures
    "mbrs_per_srvc_npi": "members per servicing NPI (high = few NPIs serving many members)",
    "setup_ratio": "setup codes / total management codes",
}

# compute benchmarks from TINs with 25+ members (same population used for signal thresholds)
big = df.query("total_mbrs >= 25")
benchmarks = (
    pd.DataFrame({
        "metric": benchmark_metrics.keys(),
        "description": benchmark_metrics.values(),
        "peer_median": [big[m].median() for m in benchmark_metrics],
        "peer_p95": [big[m].quantile(0.95) for m in benchmark_metrics],
        "peer_mean": [big[m].mean() for m in benchmark_metrics],
    })
    .round(4)
)

print(f"Benchmarks computed from {len(big)} TINs with 25+ members\n")
benchmarks

Benchmarks computed from 2105 TINs with 25+ members



,metric,description,peer_median,peer_p95,peer_mean
0,pct_no_prior_rel,% members with no prior relationship at this TIN,0.2125,0.9720,0.3029
1,pct_multi_practice_overlap,% members also receiving RPM from 2+ other TINs,0.0000,0.0645,0.0162
2,peak_mom_growth_pct,largest month-over-month member growth (%),218.3333,2000.0000,632.6518
3,growth_ratio,ratio of peak month members to first month mem...,2.0000,25.3333,7.9677
4,top_dom_pct,highest single day-of-month share of all claim...,0.3602,0.6730,0.3537
5,weekend_pct,% of claim lines on Saturday/Sunday,0.2761,0.3609,0.2521
6,pct_99458_addon,99458 add-on lines / 99457 base lines (>1 = up...,0.4675,0.9750,0.4780
7,proc_hhi,"procedure code concentration (HHI, 0-1; 1 = on...",0.3331,0.8299,0.3852
8,pct_full_bundle,% of members billed all 4 RPM codes (setup+edu...,0.2279,0.7406,0.2889
9,single_month_churn_rate,% of members active for exactly 1 month,0.1370,0.5589,0.1918


In [60]:
## Build evidence columns: for each signal, show the actual metric value and how it
## compares to the peer p95 threshold. This is what you'd cite in a narrative.
##
## Format: "evidence_<signal>" contains a short string like "72% no-prior-rel (p95 = 34%)"
## so a reader can see the number and the benchmark in one glance.

p95 = benchmarks.set_index("metric")["peer_p95"]
med = benchmarks.set_index("metric")["peer_median"]

def pct(val):
    """Format as percentage string, handling NaN."""
    if pd.isna(val):
        return "N/A"
    return f"{val:.0%}" if val <= 1 else f"{val:.0f}%"

def dec(val, places = 2):
    if pd.isna(val):
        return "N/A"
    return f"{val:.{places}f}"

df = df.assign(
    # --- sig_no_prior ---
    # "68% of members had no prior relationship (peer p95: 34%)"
    evidence_no_prior_rel = lambda x: x.pct_no_prior_rel.apply(
        lambda v: f"{pct(v)} no-prior-rel (p95={pct(p95['pct_no_prior_rel'])})"),

    # --- sig_mbr_multi_tin ---
    # "42% of members billed by 3+ TINs (p95=15%)"
    evidence_mbr_multi_tin = lambda x: x.pct_multi_practice_overlap.apply(
        lambda v: f"{pct(v)} multi-TIN members (p95={pct(p95['pct_multi_practice_overlap'])})"),

    # --- sig_ramp ---
    # "450% peak MoM growth (peer p95: 120%)"
    evidence_ramp = lambda x: x.peak_mom_growth_pct.apply(
        lambda v: f"{pct(v)} peak MoM growth (p95={pct(p95['peak_mom_growth_pct'])})"),

    # --- sig_billing_conc ---
    # "42% of lines on one day (p95=28%) | 15% weekend (p95=8%)"
    evidence_billing_conc = lambda x: (
        x.top_dom_pct.apply(lambda v: f"top-day={pct(v)}") + " (p95=" + pct(p95["top_dom_pct"]) + ") | " +
        x.weekend_pct.apply(lambda v: f"wknd={pct(v)}") + " (p95=" + pct(p95["weekend_pct"]) + ")"
    ),

    # --- sig_cpt_bundle ---
    # "addon ratio=1.4 (p95=0.8) | HHI=0.72 (p95=0.55)"
    evidence_cpt_bundle = lambda x: (
        x.pct_99458_addon.apply(lambda v: f"addon-ratio={dec(v)}") + " (p95=" + dec(p95["pct_99458_addon"]) + ") | " +
        x.proc_hhi.apply(lambda v: f"HHI={dec(v)}") + " (p95=" + dec(p95["proc_hhi"]) + ")"
    ),

    # --- sig_churn ---
    # "65% single-month churn (p95=40%)"
    evidence_churn = lambda x: x.single_month_churn_rate.apply(
        lambda v: f"{pct(v)} single-mo churn (p95={pct(p95['single_month_churn_rate'])})"),

    # --- sig_geo ---
    # "entropy=2.1 (p95=0.8) | 72% state mismatch (p95=15%)"
    evidence_geo = lambda x: (
        x.member_state_entropy.apply(lambda v: f"entropy={dec(v)}") + " (p95=" + dec(p95["member_state_entropy"]) + ") | " +
        x.pct_member_state_mismatch.apply(lambda v: f"state-mismatch={pct(v)}") + " (p95=" + pct(p95["pct_member_state_mismatch"]) + ")"
    ),

    # --- sig_bill_order ---
    # "82% same-day setup+mgmt (p95=45%) | 5% mgmt-before-setup (p90=2%)"
    evidence_bill_order = lambda x: (
        x.pct_sameday_setup_mgmt.apply(lambda v: f"same-day={pct(v)}") + " (p95=" + pct(p95["pct_sameday_setup_mgmt"]) + ") | " +
        x.pct_mgmt_before_setup.apply(lambda v: f"mgmt-before={pct(v)}")
    ),

    # --- sig_no_treat ---
    # "85% of members never received treatment mgmt (p95=60%)"
    evidence_no_treat = lambda x: x.pct_no_treatment_mgmt.apply(
        lambda v: f"{pct(v)} no-treatment-mgmt (p95={pct(p95['pct_no_treatment_mgmt'])})"),

    # --- sig_tin_network ---
    # "jaccard=0.35 (p95=0.12) | 60% shared NPIs (p95=25%)"
    evidence_tin_network = lambda x: (
        x.max_jaccard.apply(lambda v: f"jaccard={dec(v)}") + " (p95=" + dec(p95["max_jaccard"]) + ") | " +
        x.pct_shared_npis.apply(lambda v: f"shared-NPIs={pct(v)}") + " (p95=" + pct(p95["pct_shared_npis"]) + ")"
    ),

    # --- sig_peer_zscore ---
    # "3 features with z>3"
    evidence_peer_zscore = lambda x: x.n_z_flags.apply(
        lambda v: f"{v} features with z>3"),

    # --- sig_isolation ---
    # "iso score = -0.08 (flagged at <0)"
    evidence_isolation = lambda x: x.iso_score.apply(
        lambda v: f"iso-score={v:.3f} ({'flagged' if v < 0 else 'normal'})"),
)

evidence_cols = [c for c in df.columns if c.startswith("evidence_")]
print(f"Added {len(evidence_cols)} evidence columns: {evidence_cols}")
df[["prov_tin", "sample_full_nm", "priority_tier_all"] + evidence_cols].query("priority_tier_all == 'High'").head(3)

Added 12 evidence columns: ['evidence_no_prior_rel', 'evidence_mbr_multi_tin', 'evidence_ramp', 'evidence_billing_conc', 'evidence_cpt_bundle', 'evidence_churn', 'evidence_geo', 'evidence_bill_order', 'evidence_no_treat', 'evidence_tin_network', 'evidence_peer_zscore', 'evidence_isolation']


,prov_tin,sample_full_nm,priority_tier_all,evidence_no_prior_rel,evidence_mbr_multi_tin,evidence_ramp,evidence_billing_conc,evidence_cpt_bundle,evidence_churn,evidence_geo,evidence_bill_order,evidence_no_treat,evidence_tin_network,evidence_peer_zscore,evidence_isolation
24,800075306,HENDERSON PULMONARY & SLEEP MEDICINE PLL,High,63% no-prior-rel (p95=95%),6% multi-TIN members (p95=5%),1500% peak MoM growth (p95=1600%),top-day=49% (p95=71%) | wknd=1% (p95=37%),addon-ratio=N/A (p95=0.98) | HHI=1.00 (p95=0.74),24% single-mo churn (p95=53%),entropy=0.10 (p95=0.71) | state-mismatch=2% (p...,same-day=N/A (p95=100%) | mgmt-before=N/A,90% no-treatment-mgmt (p95=53%),jaccard=0.52 (p95=0.56) | shared-NPIs=100% (p9...,0 features with z>3,iso-score=-0.001 (flagged)
49,362913108,ADVOCATE HOME HEALTH SERVICE INC,High,100% no-prior-rel (p95=95%),0% multi-TIN members (p95=5%),200% peak MoM growth (p95=1600%),top-day=8% (p95=71%) | wknd=25% (p95=37%),addon-ratio=N/A (p95=0.98) | HHI=1.00 (p95=0.74),46% single-mo churn (p95=53%),entropy=-0.00 (p95=0.71) | state-mismatch=0% (...,same-day=N/A (p95=100%) | mgmt-before=N/A,0% no-treatment-mgmt (p95=53%),jaccard=N/A (p95=0.56) | shared-NPIs=100% (p95...,0 features with z>3,iso-score=-0.003 (flagged)
56,871568225,"ARANDI, M.D., NAVID R.",High,100% no-prior-rel (p95=95%),0% multi-TIN members (p95=5%),38800% peak MoM growth (p95=1600%),top-day=28% (p95=71%) | wknd=32% (p95=37%),addon-ratio=0.17 (p95=0.98) | HHI=0.70 (p95=0.74),38% single-mo churn (p95=53%),entropy=1.69 (p95=0.71) | state-mismatch=93% (...,same-day=8% (p95=100%) | mgmt-before=15%,76% no-treatment-mgmt (p95=53%),jaccard=N/A (p95=0.56) | shared-NPIs=36% (p95=...,4 features with z>3,iso-score=-0.151 (flagged)


In [61]:
## Top investigation targets with evidence
## Each row shows the provider, which signals fired, and the actual metric values
## so you can write a narrative like:
##   "Piedmont Health System (TIN 262824776, GA) — 491 members, 6 domains flagged.
##    68% of members had no prior relationship (vs 34% p95). Peak MoM growth was 450%.
##    Claims clustered on one calendar day (42% vs 28% p95). 82% of members had setup
##    and management billed same-day (clinically implausible). Members spread across
##    12 states (entropy 2.1 vs 0.8 p95). Isolation forest and peer z-score both flagged."

# signal-to-evidence mapping: only show evidence for signals that actually fired
signal_evidence_map = {
    "sig_no_prior": "evidence_no_prior_rel",
    "sig_mbr_multi_tin": "evidence_mbr_multi_tin",
    "sig_ramp": "evidence_ramp",
    "sig_billing_conc": "evidence_billing_conc",
    "sig_cpt_bundle": "evidence_cpt_bundle",
    "sig_churn": "evidence_churn",
    "sig_geo": "evidence_geo",
    "sig_bill_order": "evidence_bill_order",
    "sig_no_treat": "evidence_no_treat",
    "sig_tin_network": "evidence_tin_network",
    "sig_peer_zscore": "evidence_peer_zscore",
    "sig_isolation": "evidence_isolation",
}

def build_evidence_summary(row):
    """Combine only the evidence strings for signals that fired (=1) into one cell."""
    parts = []
    for sig, ev in signal_evidence_map.items():
        if row.get(sig, 0) == 1:
            # strip the "sig_" prefix for a cleaner label
            label = sig.replace("sig_", "").upper()
            parts.append(f"[{label}] {row[ev]}")
    return " | ".join(parts) if parts else "—"

targets = (
    df
    .query("priority_tier_all == 'High' and total_mbrs >= 25")
    .assign(evidence_summary = lambda x: x.apply(build_evidence_summary, axis = 1))
    .sort_values(["total_domains", "total_mbrs"], ascending = [False, False])
)

display_cols = [
    "prov_tin", "market", "hospital_group", "sample_full_nm",
    "total_mbrs", "total_domains", "evidence_summary"
]

print(f"{len(targets)} High-tier TINs with 25+ members\n")
targets[display_cols].head(20)

28 High-tier TINs with 25+ members



,prov_tin,market,hospital_group,sample_full_nm,total_mbrs,total_domains,evidence_summary
2085,923554980,NY,None,"FAUCI, ALEXANDER",100,5,[NO_PRIOR] 99% no-prior-rel (p95=95%) | [RAMP]...
3105,582317663,GA,ATHENS HEART CENTER,"AGRAWAL, M.D., SUBODH K.",75,5,[MBR_MULTI_TIN] 7% multi-TIN members (p95=5%) ...
133,421477471,IA,Unity Point Health,UNITYPOINT AT HOME,53,5,[NO_PRIOR] 100% no-prior-rel (p95=95%) | [CPT_...
1786,362174832,IL,Silver Cross Hospital,SILVER CROSS HOME HEALTH,35,5,[NO_PRIOR] 100% no-prior-rel (p95=95%) | [CPT_...
56,871568225,CT,None,"ARANDI, M.D., NAVID R.",3547,4,[NO_PRIOR] 100% no-prior-rel (p95=95%) | [RAMP...
4601,841894499,GA,INTEGRATED CLINIX,ERIC MILLER MD,887,4,[NO_PRIOR] 97% no-prior-rel (p95=95%) | [CPT_B...
6413,262824776,GA,Piedmont Health System,"AARON, M.D., HAYDEN KOEHLER",491,4,[RAMP] 9660% peak MoM growth (p95=1600%) | [CH...
5313,872434668,NJ,None,"MILLER, M.D., ERIC DARREN",399,4,[NO_PRIOR] 100% no-prior-rel (p95=95%) | [CPT_...
5953,752602892,TX,SLEEP MEDICINE ASSOCIATES OF TEXAS,SLEEP MEDICINE ASSOCIATES OF TEXAS PA,318,4,[CPT_BUNDLE] addon-ratio=0.00 (p95=0.98) | HHI...
208,993099556,GA,None,EXCELSIOR HEALTHCARE PARTNERS,317,4,[MBR_MULTI_TIN] 70% multi-TIN members (p95=5%)...


## 7. Export

In [101]:
## Export includes all original columns + evidence columns + evidence_summary
## The evidence_summary column is a one-line string per provider with all fired signals
## and their metric values, ready for you to expand into a narrative.
df = df.assign(evidence_summary = lambda x: x.apply(build_evidence_summary, axis = 1))

out_path = r"C:\Users\knguy139\Documents\Projects\Data\Output\kn_rpm_202608_fraud_scores.csv"
df.to_csv(out_path, index = False)
print(f"Exported {len(df)} TINs to {out_path}")

engine.dispose()

PermissionError: [Errno 13] Permission denied: 'C:\\Users\\knguy139\\Documents\\Projects\\Data\\Output\\kn_rpm_202608_fraud_scores.csv'

In [ ]:
## Save/load checkpoint — skip the full notebook rerun
## Save: run this cell after the notebook completes
## Load: run only the imports cell (cell 1), then this cell with mode = "load"

import pickle
checkpoint_path = r"C:\Users\knguy139\Documents\Projects\Data\Output\rpm_fraud_checkpoint.pkl"

mode = "save"  # change to "load" to restore

if mode == "save":
    checkpoint = {
        "df": df,
        "z_df": z_df,
        "z_features": z_features,
        "sql_signals": sql_signals,
        "all_signals": all_signals,
        "iso": iso,
        "X_scaled": X_scaled,
    }
    with open(checkpoint_path, "wb") as f:
        pickle.dump(checkpoint, f)
    print(f"Saved checkpoint ({len(df)} TINs, {df.shape[1]} cols) to {checkpoint_path}")

elif mode == "load":
    with open(checkpoint_path, "rb") as f:
        checkpoint = pickle.load(f)
    df = checkpoint["df"]
    z_df = checkpoint["z_df"]
    z_features = checkpoint["z_features"]
    sql_signals = checkpoint["sql_signals"]
    all_signals = checkpoint["all_signals"]
    iso = checkpoint["iso"]
    X_scaled = checkpoint["X_scaled"]
    print(f"Loaded checkpoint ({len(df)} TINs, {df.shape[1]} cols) from {checkpoint_path}")